# Introduction to Geospatial Data

This notebook introduces the core concepts you need to work with geospatial data in Python: **Coordinate Reference Systems (CRS)**, **vector data**, and **raster data**. Understanding these fundamentals will help you visualize, analyze, and build machine learning models on geographic information.

## 1. Coordinate Reference Systems (CRS)

A **Coordinate Reference System (CRS)** defines how coordinates (numbers) map to real-world locations on Earth. Without a CRS, coordinates are just numbers with no geographic meaning.

### Geographic vs Projected CRS

- **Geographic CRS** (e.g., WGS84): Uses latitude and longitude in degrees. Good for storing data and global analysis, but distances and areas are distorted.
- **Projected CRS** (e.g., UTM): Flattens the Earth onto a 2D plane. Better for measuring distances, areas, and for regional maps. Each UTM zone covers a narrow strip of longitude.

### Why Reprojection Matters

When combining datasets (e.g., overlaying vector buildings on a raster image), **all layers must share the same CRS**. Otherwise, they will not align. Reprojection converts coordinates from one CRS to another.

In [ ]:
# Optional: Install packages if needed
# %pip install geopandas rasterio pyproj

import geopandas as gpd
from shapely.geometry import Point

# Create a point in WGS84 (latitude, longitude)
point_wgs84 = Point(-122.4194, 37.7749)  # San Francisco
gdf = gpd.GeoDataFrame({"name": ["SF"]}, geometry=[point_wgs84], crs="EPSG:4326")
print("CRS:", gdf.crs)
print("Bounds (lon, lat):", gdf.total_bounds)

In [ ]:
# Reproject to a projected CRS (UTM zone 10N) for distance/area calculations
gdf_projected = gdf.to_crs("EPSG:32610")
print("Reprojected CRS:", gdf_projected.crs)
print("Bounds (meters):", gdf_projected.total_bounds)

## 2. Vector Data

**Vector data** represents geographic features as discrete geometries: **points**, **lines**, and **polygons**. Each feature has attributes (e.g., name, population) stored in a table.

### Geometry Types

- **Points**: Single locations (e.g., cities, sensors)
- **Lines**: Linear features (e.g., roads, rivers)
- **Polygons**: Enclosed areas (e.g., buildings, countries, land parcels)

### Common Formats

- **GeoJSON**: JSON-based, human-readable, widely used on the web
- **Shapefile**: Classic format (.shp + .dbf + .shx); one geometry type per file
- **GPKG (GeoPackage)**: Modern, single-file format; supports multiple layers

In [ ]:
from shapely.geometry import Point, LineString, Polygon

# Example geometries
point = Point(0, 0)
line = LineString([(0, 0), (1, 1), (2, 0)])
polygon = Polygon([(0, 0), (1, 0), (1, 1), (0, 1), (0, 0)])

# GeoDataFrame can hold any of these
gdf = gpd.GeoDataFrame(
    {"type": ["point", "line", "polygon"]},
    geometry=[point, line, polygon],
    crs="EPSG:4326"
)
print(gdf)

## 3. Raster Data

**Raster data** represents the world as a grid of cells (pixels). Each cell has one or more values. Satellite imagery, elevation models, and land cover maps are typically rasters.

### Key Concepts

- **Bands**: Layers of values (e.g., RGB = 3 bands; multispectral = many bands)
- **Resolution**: Size of each pixel in ground units (e.g., 10 m × 10 m)
- **Extent**: The geographic bounds (min/max x and y) of the raster
- **NoData**: Value used for missing or invalid cells (e.g., clouds, outside the area)

In [ ]:
import rasterio
import numpy as np
from pathlib import Path

# Create a minimal example raster to demonstrate structure (works offline)
data_dir = Path("../data")
data_dir.mkdir(exist_ok=True)
demo_raster = data_dir / "demo_raster.tif"

# Write a small 10x10 single-band raster
arr = np.random.randint(0, 255, (10, 10), dtype=np.uint8)
transform = rasterio.transform.from_bounds(0, 0, 100, 100, 10, 10)
with rasterio.open(
    demo_raster, "w", driver="GTiff", height=10, width=10,
    count=1, dtype=arr.dtype, crs="EPSG:4326", transform=transform
) as dst:
    dst.write(arr, 1)

# Inspect raster metadata
with rasterio.open(demo_raster) as src:
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Resolution:", src.res)
    print("Number of bands:", src.count)
    print("Width x Height:", src.width, "x", src.height)

## Recap

| Concept | Summary |
|---------|---------|
| **CRS** | Defines how coordinates map to Earth; geographic (lat/lon) vs projected (meters). Always match CRS when combining layers. |
| **Vector** | Points, lines, polygons with attributes. Formats: GeoJSON, Shapefile, GeoPackage. |
| **Raster** | Grid of cells with bands, resolution, extent, and NoData. Used for imagery and continuous surfaces. |

## Check Your Understanding

1. **CRS**: Why would you reproject a dataset from WGS84 to UTM before computing the area of a polygon?
2. **Vector**: What geometry type would you use to represent a road? A city? A lake?
3. **Raster**: If a satellite image has 10 m resolution, what does that mean for each pixel?